# Dental X-Ray Machine Learning Project
Supervised Learning • Regression + Classification + Cross Validation

In [ ]:

!pip install -q kaggle numpy pandas scikit-learn matplotlib seaborn pillow


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image

from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, classification_report, confusion_matrix


### Load Dental Radiography Dataset from Kaggle

In [ ]:

from google.colab import files
print("Upload kaggle.json")
uploaded = files.upload()

import os, shutil, zipfile

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d imtkaggleteam/dental-radiography -p /content
zip_path='/content/dental-radiography.zip'

DATA_PATH='/content/dental_radiography'
os.makedirs(DATA_PATH,exist_ok=True)
with zipfile.ZipFile(zip_path,'r') as z:
    z.extractall(DATA_PATH)

print('Dataset loaded ✓')


Upload kaggle.json


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/imtkaggleteam/dental-radiography
License(s): CC-BY-SA-4.0
dental-radiography.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset loaded ✓


### Parse XML annotations & extract cavity labels

In [ ]:

IMG_SIZE=(128,128)
X=[]; y_reg=[]; y_clf=[]

root=Path(DATA_PATH)
xml_files=list(root.rglob("*.xml"))

def extract(xml):
    tree=ET.parse(xml); r=tree.getroot()
    cavity=sum(1 for o in r.findall("object") if o.find('name').text=='Cavity')
    return cavity,1 if cavity>0 else 0

def img_from_xml(xml):
    name=ET.parse(xml).getroot().find('filename').text
    p=list(root.rglob(name))[0]
    img=Image.open(p).convert("L").resize(IMG_SIZE)
    return np.array(img)/255.0

for f in xml_files[:400]:  # limited for speed
    try:
        im=img_from_xml(f)
        c,h=extract(f)
        X.append(im.flatten());
        y_reg.append(c);
        y_clf.append(h)
    except: pass

X=np.array(X); y_reg=np.array(y_reg); y_clf=np.array(y_clf)
print("Samples:",X.shape," Labels:",len(y_reg))


Samples: (0,)  Labels: 0


### Train/Test split

In [ ]:

X_train,X_test,y_reg_train,y_reg_test,y_clf_train,y_clf_test=train_test_split(
    X,y_reg,y_clf,test_size=0.2,random_state=42,stratify=y_clf
)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## Predict number of cavities using Linear Regression

In [ ]:

lr=LinearRegression()
lr.fit(X_train,y_reg_train)

pred=lr.predict(X_test)
print("MAE:",mean_absolute_error(y_reg_test,pred)," R²:",r2_score(y_reg_test,pred))

kf=KFold(5,shuffle=True,random_state=42)
cv_mae=-cross_val_score(lr,X,y_reg,cv=kf,scoring='neg_mean_absolute_error')
cv_r2=cross_val_score(lr,X,y_reg,cv=kf,scoring='r2')

print("
5-Fold CV Regression Results")
print("MAE:",cv_mae.mean()," R²:",cv_r2.mean())


## Classification: Compare multiple models with CV

In [ ]:

models={
"Logistic Regression":LogisticRegression(max_iter=2000),
"Random Forest":RandomForestClassifier(n_estimators=200),
"SVM RBF":SVC(kernel='rbf')
}

skf=StratifiedKFold(5,shuffle=True,random_state=42)

results={}
for name,m in models.items():
    f1=cross_val_score(m,X,y_clf,cv=skf,scoring='f1').mean()
    results[name]=f1

results


### Train best model on full train set & evaluate on test

In [ ]:

best=RandomForestClassifier(n_estimators=200).fit(X_train,y_clf_train)
yp=best.predict(X_test)

cm=confusion_matrix(y_clf_test,yp)
sns.heatmap(cm,annot=True,cmap="Blues"); plt.show()

print(classification_report(y_clf_test,yp))
